# LeetCode 3219 — Minimum Cost for Cutting Cake II bằng Python

- Link: https://leetcode.com/problems/minimum-cost-for-cutting-cake-ii/
- Độ khó: **Hard**
- Chủ đề: **Tham lam** — cắt đường **đắt nhất trước**; mỗi lần cắt nhân với số mảnh theo chiều còn lại. Chứng minh bằng **đổi chỗ (exchange argument)**; công thức cặp `Σh + Σv + Σ min(h, v)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumCost(self, m: int, n: int, horizontalCut: List[int], verticalCut: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu rằng một đường cắt ngang `i` sẽ phải cắt **lặp lại** trên mọi cột-mảnh đang tồn tại: chi phí thực `= horizontalCut[i] × (số mảnh theo chiều dọc lúc đó)`.
2. Thấy trật tự cắt chỉ quan trọng **giữa hai loại**: cắt đường đắt trước để nó bị nhân với hệ số nhỏ.
3. Chứng minh tham lam bằng đổi chỗ hai bước kề nhau.
4. Viết lại tổng chi phí thành `Σh + Σv + Σ_{(h, v)} min(h, v)` — mỗi cặp (ngang, dọc) đóng góp đường **rẻ hơn** một lần nữa.
5. Kiểm chứng bằng DP trên hình chữ nhật con (brute).

## 2. Đề bài & ví dụ

Bánh `m × n` cần cắt thành các ô `1 × 1`. `horizontalCut` (độ dài `m − 1`) cho chi phí cắt theo đường ngang `i`; `verticalCut` (độ dài `n − 1`) cho chi phí đường dọc `j`. Mỗi thao tác chọn một mảnh chưa phải `1 × 1` và cắt nó theo một đường ngang/dọc đi qua mảnh, chi phí bằng chi phí gốc của đường đó (không đổi theo kích thước mảnh). Tìm tổng chi phí nhỏ nhất.

**Ví dụ 1:** `m = 3, n = 2, horizontalCut = [1, 3], verticalCut = [5]` → `13`.

**Ví dụ 2:** `m = 2, n = 2, horizontalCut = [7], verticalCut = [4]` → `15`.

**Ràng buộc:** `1 ≤ m, n ≤ 10^5`; `1 ≤ horizontalCut[i], verticalCut[j] ≤ 10^3`.

In [1]:
from typing import List
from functools import lru_cache
import heapq, random, time

examples = [(3, 2, [1, 3], [5], 13), (2, 2, [7], [4], 15)]
for m, n, H, V, exp in examples:
    print(f'm={m}, n={n}, H={H}, V={V} -> kỳ vọng {exp}')

m=3, n=2, H=[1, 3], V=[5] -> kỳ vọng 13
m=2, n=2, H=[7], V=[4] -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. Đếm số lần một đường bị cắt

Gọi `hp` = số mảnh theo chiều **ngang** (số dải hàng) và `vp` = số mảnh theo chiều **dọc** (số dải cột). Ban đầu `hp = vp = 1`.

- Cắt trọn đường ngang `i` (trên mọi mảnh nó đi qua) cần đúng `vp` thao tác ⇒ chi phí `H[i] · vp`, sau đó `hp += 1`.
- Cắt trọn đường dọc `j` cần `hp` thao tác ⇒ chi phí `V[j] · hp`, sau đó `vp += 1`.

Trong lời giải tối ưu có thể coi mỗi đường được cắt "trọn" một lượt (các mảnh độc lập, xen kẽ không làm giảm chi phí) — DP brute ở mục 5 kiểm chứng điều này.

In [2]:
def cost_of_order(order, H, V):
    """order: danh sách ('h', i) / ('v', j) — mỗi đường cắt trọn một lượt."""
    hp = vp = 1; total = 0
    for kind, idx in order:
        if kind == 'h':
            total += H[idx] * vp; hp += 1
        else:
            total += V[idx] * hp; vp += 1
    return total

print(cost_of_order([('v', 0), ('h', 0), ('h', 1)], [1, 3], [5]))   # 5 + 1*2 + 3*2 = 13
print(cost_of_order([('h', 0), ('h', 1), ('v', 0)], [1, 3], [5]))   # 1 + 3 + 5*3 = 19

13
19


### 3.2. Exchange argument (đổi chỗ hai bước kề nhau)

Xét hai bước liên tiếp: ngang `h` rồi dọc `v`, trạng thái trước là `(hp, vp)`:

- `h` trước: `h·vp + v·(hp+1)`.
- `v` trước: `v·hp + h·(vp+1)`.

Hiệu (h trước − v trước) `= v − h`. Vậy nên đặt đường **đắt hơn** trước. Hai bước cùng loại đổi chỗ không đổi chi phí. ⇒ Sắp tất cả đường giảm dần theo chi phí là tối ưu.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Một đường cắt "đắt" mà để muộn sẽ bị nhân với số mảnh lớn — giống xếp lịch: việc nặng làm sớm khi hệ số còn nhỏ. Ở ví dụ 1, đường dọc giá `5` cắt trước (nhân 1), hai đường ngang rẻ cắt sau (nhân 2): `5 + 2 + 6 = 13`; nếu cắt ngang trước, đường `5` bị nhân 3 ⇒ `19`.

## 5. Cách 1 — Brute Force: DP trên hình chữ nhật con

Trạng thái là mảnh `[r1, r2) × [c1, c2)`; thử mọi đường cắt đi qua mảnh. Đây là mô hình **đúng nghĩa đề bài** (cắt từng mảnh độc lập), không giả sử "cắt trọn đường".

In [3]:
def brute(m, n, H, V):
    @lru_cache(maxsize=None)
    def f(r1, r2, c1, c2):
        if r2 - r1 == 1 and c2 - c1 == 1:
            return 0
        best = float('inf')
        for i in range(r1, r2 - 1):            # đường ngang i nằm giữa hàng i và i+1
            best = min(best, H[i] + f(r1, i + 1, c1, c2) + f(i + 1, r2, c1, c2))
        for j in range(c1, c2 - 1):
            best = min(best, V[j] + f(r1, r2, c1, j + 1) + f(r1, r2, j + 1, c2))
        return best
    return f(0, m, 0, n)

for m, n, H, V, exp in examples:
    print('brute:', brute(m, n, H, V), '| kỳ vọng', exp)

brute: 13 | kỳ vọng 13
brute: 15 | kỳ vọng 15


**Độ phức tạp:** `O(m² n² (m + n))` — chỉ cho `m, n ≤ 6`.

Ta cũng kiểm thêm mô hình "cắt trọn đường": thử **mọi hoán vị** thứ tự các đường.

In [4]:
import itertools
def brute_orders(m, n, H, V):
    lines = [('h', i) for i in range(m - 1)] + [('v', j) for j in range(n - 1)]
    return min(cost_of_order(p, H, V) for p in itertools.permutations(lines)) if lines else 0

rng = random.Random(3219)
for _ in range(150):
    m = rng.randint(1, 4); n = rng.randint(1, 4)
    H = [rng.randint(1, 9) for _ in range(m - 1)]; V = [rng.randint(1, 9) for _ in range(n - 1)]
    assert brute(m, n, H, V) == brute_orders(m, n, H, V)
print('DP hình chữ nhật con = tối ưu trên các thứ tự "cắt trọn đường" (150 ca).')

DP hình chữ nhật con = tối ưu trên các thứ tự "cắt trọn đường" (150 ca).


## 6. Ý tưởng thuật toán / bước đột phá

```
sắp H, V giảm dần; hp = vp = 1
trộn hai dãy như merge sort, mỗi bước lấy phần tử lớn hơn:
    lấy h:  ans += h * vp; hp += 1
    lấy v:  ans += v * hp; vp += 1
```

`O(m log m + n log n)`. Với chi phí `≤ 10³` có thể thay sắp xếp bằng **đếm** (counting sort) ⇒ `O(m + n + 10³)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam sắp xếp + trộn hai con trỏ (lời giải chính)

In [5]:
def greedy_merge(m, n, H, V):
    H = sorted(H, reverse=True); V = sorted(V, reverse=True)
    i = j = 0; hp = vp = 1; ans = 0
    while i < len(H) or j < len(V):
        if j == len(V) or (i < len(H) and H[i] >= V[j]):
            ans += H[i] * vp; hp += 1; i += 1
        else:
            ans += V[j] * hp; vp += 1; j += 1
    return ans

for m, n, H, V, exp in examples:
    print('merge:', greedy_merge(m, n, H, V), '| kỳ vọng', exp)

merge: 13 | kỳ vọng 13
merge: 15 | kỳ vọng 15


### 7.2. Cách 3 — Tham lam bằng một heap chung

In [6]:
def greedy_heap(m, n, H, V):
    heap = [(-c, 'h') for c in H] + [(-c, 'v') for c in V]
    heapq.heapify(heap)
    hp = vp = 1; ans = 0
    while heap:
        c, kind = heapq.heappop(heap); c = -c
        if kind == 'h':
            ans += c * vp; hp += 1
        else:
            ans += c * hp; vp += 1
    return ans

for m, n, H, V, exp in examples:
    print('heap:', greedy_heap(m, n, H, V), '| kỳ vọng', exp)

heap: 13 | kỳ vọng 13
heap: 15 | kỳ vọng 15


### 7.3. Cách 4 — Công thức cặp: `Σh + Σv + Σ_{h, v} min(h, v)`

Mỗi đường bị cắt `1 + (số đường loại kia cắt trước nó)` lần. Với thứ tự giảm dần, trong mỗi cặp (ngang `h`, dọc `v`), đường **rẻ hơn** được cắt sau và bị tính thêm đúng một lần vì đường kia ⇒ cặp đóng góp `min(h, v)`. Tính `Σ min` bằng sắp xếp + tổng tiền tố + `bisect`.

In [7]:
from bisect import bisect_left
def pair_formula(m, n, H, V):
    Vs = sorted(V); pre = [0]
    for x in Vs:
        pre.append(pre[-1] + x)
    total = sum(H) + sum(V)
    for h in H:
        k = bisect_left(Vs, h)                    # các v < h: min = v ; còn lại min = h
        total += pre[k] + h * (len(Vs) - k)
    return total

for m, n, H, V, exp in examples:
    print('pair:', pair_formula(m, n, H, V), '| kỳ vọng', exp)

pair: 13 | kỳ vọng 13
pair: 15 | kỳ vọng 15


### 7.4. Cách 5 — Counting sort vì chi phí `≤ 10³`

In [8]:
def counting(m, n, H, V, C=1000):
    ch = [0] * (C + 1); cv = [0] * (C + 1)
    for x in H: ch[x] += 1
    for x in V: cv[x] += 1
    hp = vp = 1; ans = 0
    for c in range(C, 0, -1):
        # cùng giá: thứ tự giữa ngang và dọc không ảnh hưởng (hiệu = v - h = 0)
        if ch[c]:
            ans += c * vp * ch[c]; hp += ch[c]
        if cv[c]:
            ans += c * hp * cv[c]; vp += cv[c]
    return ans

for m, n, H, V, exp in examples:
    print('counting:', counting(m, n, H, V), '| kỳ vọng', exp)

counting: 13 | kỳ vọng 13
counting: 15 | kỳ vọng 15


## 8. TRACE từng bước

Trace cách 2 trên `m = 4, n = 3, H = [2, 6, 3], V = [4, 1]`.

In [9]:
def trace(m, n, H, V):
    Hs = sorted(H, reverse=True); Vs = sorted(V, reverse=True)
    i = j = 0; hp = vp = 1; ans = 0
    while i < len(Hs) or j < len(Vs):
        if j == len(Vs) or (i < len(Hs) and Hs[i] >= Vs[j]):
            add = Hs[i] * vp
            print(f'cắt ngang {Hs[i]} trên {vp} dải cột -> +{add}', end='')
            ans += add; hp += 1; i += 1
        else:
            add = Vs[j] * hp
            print(f'cắt dọc  {Vs[j]} trên {hp} dải hàng -> +{add}', end='')
            ans += add; vp += 1; j += 1
        print(f' | (hp, vp) = ({hp}, {vp}) | tổng {ans}')
    print('đáp án:', ans, '| brute:', brute(m, n, H, V), '| công thức cặp:', pair_formula(m, n, H, V))

trace(4, 3, [2, 6, 3], [4, 1])

cắt ngang 6 trên 1 dải cột -> +6 | (hp, vp) = (2, 1) | tổng 6
cắt dọc  4 trên 2 dải hàng -> +8 | (hp, vp) = (2, 2) | tổng 14
cắt ngang 3 trên 2 dải cột -> +6 | (hp, vp) = (3, 2) | tổng 20
cắt ngang 2 trên 2 dải cột -> +4 | (hp, vp) = (4, 2) | tổng 24
cắt dọc  1 trên 4 dải hàng -> +4 | (hp, vp) = (4, 3) | tổng 28
đáp án: 28 | brute: 28 | công thức cặp: 28


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minimumCost(self, m: int, n: int, horizontalCut: List[int], verticalCut: List[int]) -> int:
        H = sorted(horizontalCut, reverse=True)
        V = sorted(verticalCut, reverse=True)
        i = j = 0
        hp = vp = 1
        ans = 0
        lh, lv = len(H), len(V)
        while i < lh or j < lv:
            if j == lv or (i < lh and H[i] >= V[j]):
                ans += H[i] * vp
                hp += 1
                i += 1
            else:
                ans += V[j] * hp
                vp += 1
                j += 1
        return ans

for m, n, H, V, exp in examples:
    got = Solution().minimumCost(m, n, H, V)
    print(got, 'OK' if got == exp else 'SAI')

13 OK
15 OK


## 10. Vì sao thuật toán đúng

1. **Quy về thứ tự đường:** mọi lời giải có thể biến đổi (không tăng chi phí) thành lời giải "cắt trọn" từng đường theo một thứ tự; khi đó chi phí `= Σ_h h·(1 + #v trước h) + Σ_v v·(1 + #h trước v)`. (Mục 5 kiểm chứng DP trên hình chữ nhật con cho cùng giá trị.)
2. **Đổi chỗ:** nếu trong thứ tự có cặp kề nhau `x` rồi `y` khác loại với `cost(x) < cost(y)`, đổi chỗ giảm chi phí đúng `cost(y) − cost(x) > 0` (mục 3.2) ⇒ thứ tự tối ưu không có "nghịch thế" khác loại ⇒ là thứ tự giảm dần (cùng loại hoặc bằng nhau thì không ảnh hưởng).
3. **Công thức cặp:** trong thứ tự giảm dần, với cặp `(h, v)`, đường nhỏ hơn đứng sau nên được cộng thêm đúng một lần ⇒ `Σh + Σv + Σ min(h, v)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP hình chữ nhật con | `O(m² n² (m+n))` | `O(m² n²)` |
| 2. Sắp xếp + trộn | `O(m log m + n log n)` | `O(m + n)` |
| 3. Heap | `O((m+n) log(m+n))` | `O(m + n)` |
| 4. Công thức cặp | `O((m+n) log n)` | `O(n)` |
| 5. Counting sort | `O(m + n + C)` | `O(C)` |

Kết quả lớn nhất cỡ `10^3 · 10^5 · 10^5 = 10^{13}` — Python không tràn số; C++/Java cần `long long`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'merge': greedy_merge, 'heap': greedy_heap, 'pair': pair_formula, 'counting': counting,
         'submit': lambda m, n, H, V: Solution().minimumCost(m, n, H, V)}
edge = [(1, 1, [], [], 0), (1, 4, [], [2, 5, 1], 8), (3, 1, [4, 4], [], 8),
        (2, 2, [5], [5], 15), (2, 3, [1], [10, 10], 23), (3, 3, [1, 1], [1, 1], 8)]
for m, n, H, V, exp in examples + edge:
    assert brute(m, n, H, V) == exp, (m, n, H, V, brute(m, n, H, V))
    for name, f in impls.items():
        assert f(m, n, H, V) == exp, (name, m, n, H, V)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `1 × 1` không cần cắt (`0`); chỉ có một chiều ⇒ tổng các chi phí (mỗi đường cắt đúng một lần); `(2, 2, [5], [5])`: `5 + 5·2 = 15` bất kể thứ tự; `(2, 3, [1], [10, 10])`: hai đường dọc giá 10 cắt trước (mỗi cái nhân 1) rồi đường ngang giá 1 trên 3 dải cột ⇒ `10 + 10 + 3 = 23` — nếu cắt ngang trước thì `1 + 10·2 + 10·2 = 41`.

In [12]:
for order in ([('v', 0), ('v', 1), ('h', 0)], [('h', 0), ('v', 0), ('v', 1)]):
    print(order, '->', cost_of_order(order, [1], [10, 10]))

[('v', 0), ('v', 1), ('h', 0)] -> 23
[('h', 0), ('v', 0), ('v', 1)] -> 41


### 12.2. Đối chiếu ngẫu nhiên với DP brute

In [13]:
cnt = 0
for _ in range(300):
    m = rng.randint(1, 5); n = rng.randint(1, 5)
    H = [rng.randint(1, 8) for _ in range(m - 1)]; V = [rng.randint(1, 8) for _ in range(n - 1)]
    exp = brute(m, n, H, V)
    for name, f in impls.items():
        assert f(m, n, H, V) == exp, (name, m, n, H, V)
    cnt += 1
print('Khớp DP brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp DP brute trên 300 bộ ngẫu nhiên.


### 12.3. Ca lớn `m = n = 10^5`

In [14]:
M = N = 10 ** 5
big = {
    'ngẫu nhiên': ([rng.randint(1, 1000) for _ in range(M - 1)], [rng.randint(1, 1000) for _ in range(N - 1)]),
    'toàn 1000': ([1000] * (M - 1), [1000] * (N - 1)),
    'tăng dần': (list(range(1, M)), list(range(N - 1, 0, -1))),
}
for name, (H, V) in big.items():
    H = [min(x, 1000) for x in H]; V = [min(x, 1000) for x in V]
    t = time.time(); x = Solution().minimumCost(M, N, H, V); t1 = time.time() - t
    y = pair_formula(M, N, H, V); z = counting(M, N, H, V)
    assert x == y == z
    print(f'{name:10s}: {x:16d} | nộp {t1:.2f}s')
k = M - 1
assert Solution().minimumCost(M, N, [1000] * k, [1000] * k) == 1000 * (2 * k + k * k)
print('Toàn 1000: 1000·(2k + k²) đúng theo công thức cặp.')

ngẫu nhiên:    3341831239831 | nộp 0.07s
toàn 1000 :    9999999999000 | nộp 0.03s
tăng dần  :    9900432832500 | nộp 0.03s
Toàn 1000: 1000·(2k + k²) đúng theo công thức cặp.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cắt đường **rẻ** trước (sắp tăng dần)

In [15]:
def wrong_ascending(m, n, H, V):
    H = sorted(H); V = sorted(V)                               # SAI: tăng dần
    i = j = 0; hp = vp = 1; ans = 0
    while i < len(H) or j < len(V):
        if j == len(V) or (i < len(H) and H[i] <= V[j]):
            ans += H[i] * vp; hp += 1; i += 1
        else:
            ans += V[j] * hp; vp += 1; j += 1
    return ans

print('sai :', wrong_ascending(*examples[0][:4]), '| đúng:', Solution().minimumCost(*examples[0][:4]))

sai : 19 | đúng: 13


**Vì sao sai:** đường đắt để cuối bị nhân với số mảnh lớn nhất — đây thực ra là thứ tự **tệ nhất** (xem biến thể "chi phí lớn nhất").

### Lỗi 2 — Nhân với số mảnh **cùng chiều** thay vì chiều kia

In [16]:
def wrong_counter(m, n, H, V):
    H = sorted(H, reverse=True); V = sorted(V, reverse=True)
    i = j = 0; hp = vp = 1; ans = 0
    while i < len(H) or j < len(V):
        if j == len(V) or (i < len(H) and H[i] >= V[j]):
            ans += H[i] * hp; hp += 1; i += 1                    # SAI: phải là vp
        else:
            ans += V[j] * vp; vp += 1; j += 1                    # SAI: phải là hp
    return ans

print('sai :', wrong_counter(*examples[0][:4]), '| đúng:', Solution().minimumCost(*examples[0][:4]))

sai : 10 | đúng: 13


**Vì sao sai:** đường ngang cắt ngang qua mọi **dải cột** (`vp`), không phải qua các dải hàng.

### Lỗi 3 — Cắt hết một loại rồi mới tới loại kia

In [17]:
def wrong_type_first(m, n, H, V):
    best = float('inf')
    for first in ('h', 'v'):
        order = ([('h', i) for i in sorted(range(m - 1), key=lambda i: -H[i])] +
                 [('v', j) for j in sorted(range(n - 1), key=lambda j: -V[j])])
        if first == 'v':
            order = order[m - 1:] + order[:m - 1]
        best = min(best, cost_of_order(order, H, V))
    return best                                                   # SAI

args = (3, 3, [5, 1], [4, 2])
print('sai :', wrong_type_first(*args), '| đúng:', Solution().minimumCost(*args))

sai : 24 | đúng: 20


**Vì sao sai:** thứ tự tối ưu **đan xen** hai loại theo giá trị (`5, 4, 2, 1`): ngang–dọc–dọc–ngang. Cắt hết ngang trước buộc đường dọc `4` nhân 3.

### Lỗi 4 — Bỏ hẳn hệ số nhân (coi mỗi đường cắt một lần)

In [18]:
def wrong_no_multiplier(m, n, H, V):
    return sum(H) + sum(V)                                        # SAI

print('sai :', wrong_no_multiplier(*examples[1][:4]), '| đúng:', Solution().minimumCost(*examples[1][:4]))

sai : 11 | đúng: 15


**Vì sao sai:** sau khi cắt một đường, các mảnh tách rời — đường kia phải cắt **trên từng mảnh**. Đây chính là phần `Σ min(h, v)` bị thiếu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cắt bánh / ván gỗ, chi phí đường cắt × số mảnh" ⇒ tham lam đắt trước (bài kinh điển *Board Cutting*).
- Chứng minh tham lam bằng đổi chỗ hai bước kề: so sánh hiệu chi phí.
- Phiên bản I (LC 3218) có `m, n ≤ 20` — cho phép DP; phiên bản II buộc tham lam.

## 15. Biến thể của bài

1. **LC 3218** (bản I): ràng buộc nhỏ, cùng lời giải.
2. Chi phí **lớn nhất**: sắp tăng dần (đổi chiều exchange argument).
3. Bỏ được một đường ngang miễn phí (bài tập 5).
4. Thêm chiều thứ ba (khối hộp): mỗi lát cắt nhân với tích số mảnh của **hai** chiều còn lại — exchange argument không còn đơn giản như 2D.

In [19]:
# Biến thể 2: chi phí lớn nhất = sắp tăng dần (chính là "lỗi 1")
def brute_max(m, n, H, V):
    @lru_cache(maxsize=None)
    def f(r1, r2, c1, c2):
        if r2 - r1 == 1 and c2 - c1 == 1:
            return 0
        best = -1
        for i in range(r1, r2 - 1):
            best = max(best, H[i] + f(r1, i + 1, c1, c2) + f(i + 1, r2, c1, c2))
        for j in range(c1, c2 - 1):
            best = max(best, V[j] + f(r1, r2, c1, j + 1) + f(r1, r2, j + 1, c2))
        return best
    return f(0, m, 0, n)

for _ in range(150):
    m = rng.randint(1, 4); n = rng.randint(1, 4)
    H = [rng.randint(1, 8) for _ in range(m - 1)]; V = [rng.randint(1, 8) for _ in range(n - 1)]
    assert wrong_ascending(m, n, H, V) == brute_max(m, n, H, V)
print('Biến thể "chi phí lớn nhất": sắp tăng dần khớp DP max.')

Biến thể "chi phí lớn nhất": sắp tăng dần khớp DP max.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | DP hình chữ nhật con | đa thức bậc cao | Oracle, đúng nghĩa đề |
| 2 | Sắp xếp + trộn | `O(k log k)` | **Nộp bài** |
| 3 | Heap | `O(k log k)` | Dễ viết |
| 4 | Công thức cặp | `O(k log k)` | Không mô phỏng |
| 5 | Counting sort | `O(k + C)` | Tận dụng `C ≤ 10³` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `m = 2, n = 3, H = [3], V = [1, 2]`.

**Bài 2 (tính tay):** `m = 3, n = 3, H = [2, 2], V = [3, 1]` — viết thứ tự cắt và chi phí từng bước.

**Bài 3 (điền TODO):** trả về **thứ tự cắt** (danh sách `('h', i)` / `('v', j)` theo chỉ số gốc) đạt tối ưu.

**Bài 4 (tự viết lại):** cài lại công thức cặp `Σh + Σv + Σ min(h, v)` bằng **hai con trỏ** (sắp cả hai tăng dần), không dùng `bisect`.

**Bài 5 (mini challenge):** được bỏ (không phải cắt) **đúng một** đường ngang — các mảnh cuối có thể là `2 × 1` ở chỗ đó. Tìm chi phí nhỏ nhất, `O((m + n) log)`.

In [20]:
def cut_order_todo(m, n, H, V):
    # TODO
    raise NotImplementedError

def pair_two_pointers_todo(m, n, H, V):
    # TODO
    raise NotImplementedError

def skip_one_horizontal_todo(m, n, H, V):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Thứ tự giảm dần `3, 2, 1`. (2) `3·1 + 2·2 + 1·2`. (3) `9`.

**Bài 2.** (1) Thứ tự `3 (dọc), 2, 2 (ngang), 1 (dọc)`. (2) `3·1 + 2·2 + 2·2 + 1·3`. (3) `14`.

**Bài 3.** (1) Sắp **chỉ số** theo chi phí giảm dần. (2) Trộn hai danh sách chỉ số. (3) Kiểm bằng `cost_of_order`.

**Bài 4.** (1) Với `h` tăng dần, số `v < h` (và tổng của chúng) chỉ tăng ⇒ con trỏ `k` trên `V` chỉ tiến. (2) Đóng góp của `h`: `tổng(v < h) + h · (số v ≥ h)`. (3) Kiểm với `pair_formula`.

**Bài 5.** (1) Bỏ đường ngang `h` = bài gốc trên `H \ {h}`. (2) Theo công thức cặp, chi phí giảm `h + Σ_v min(h, v)`. (3) Chọn `h` có mức giảm lớn nhất — chính là `max(H)` vì mức giảm tăng theo `h`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
assert Solution().minimumCost(2, 3, [3], [1, 2]) == 9
assert Solution().minimumCost(3, 3, [2, 2], [3, 1]) == 14

def cut_order(m, n, H, V):
    hi = sorted(range(m - 1), key=lambda i: -H[i]); vi = sorted(range(n - 1), key=lambda j: -V[j])
    a = b = 0; order = []
    while a < len(hi) or b < len(vi):
        if b == len(vi) or (a < len(hi) and H[hi[a]] >= V[vi[b]]):
            order.append(('h', hi[a])); a += 1
        else:
            order.append(('v', vi[b])); b += 1
    return order

for _ in range(200):
    m = rng.randint(1, 6); n = rng.randint(1, 6)
    H = [rng.randint(1, 9) for _ in range(m - 1)]; V = [rng.randint(1, 9) for _ in range(n - 1)]
    od = cut_order(m, n, H, V)
    assert sorted(od) == sorted([('h', i) for i in range(m - 1)] + [('v', j) for j in range(n - 1)])
    assert cost_of_order(od, H, V) == Solution().minimumCost(m, n, H, V)
print('Bài 1: 9 | Bài 2: 14 | Bài 3: thứ tự cắt hợp lệ và tối ưu. Ví dụ 1:', cut_order(3, 2, [1, 3], [5]))

Bài 1: 9 | Bài 2: 14 | Bài 3: thứ tự cắt hợp lệ và tối ưu. Ví dụ 1: [('v', 0), ('h', 1), ('h', 0)]


In [22]:
# Bài 4, 5
def pair_two_pointers(m, n, H, V):
    Hs = sorted(H); Vs = sorted(V)
    total = sum(H) + sum(V); k = 0; small = 0
    for h in Hs:
        while k < len(Vs) and Vs[k] < h:
            small += Vs[k]; k += 1
        total += small + h * (len(Vs) - k)
    return total

def skip_one_horizontal(m, n, H, V):
    if not H:
        return Solution().minimumCost(m, n, H, V)
    base = pair_formula(m, n, H, V)
    h = max(H)
    return base - h - sum(min(h, v) for v in V)

for _ in range(120):
    m = rng.randint(1, 4); n = rng.randint(1, 4)
    H = [rng.randint(1, 3) for _ in range(m - 1)]; V = [rng.randint(1, 3) for _ in range(n - 1)]
    assert pair_two_pointers(m, n, H, V) == pair_formula(m, n, H, V) == brute(m, n, H, V)
    if H:
        exp = min(Solution().minimumCost(m - 1, n, H[:i] + H[i + 1:], V) for i in range(len(H)))
        assert skip_one_horizontal(m, n, H, V) == exp
print('Bài 4: hai con trỏ khớp công thức cặp và brute | Bài 5: bỏ đường ngang lớn nhất là tối ưu.')

Bài 4: hai con trỏ khớp công thức cặp và brute | Bài 5: bỏ đường ngang lớn nhất là tối ưu.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
H.sort(reverse=True); V.sort(reverse=True)
i = j = 0; hp = vp = 1; ans = 0
while i < len(H) or j < len(V):
    if j == len(V) or (i < len(H) and H[i] >= V[j]):
        ans += H[i] * vp; hp += 1; i += 1
    else:
        ans += V[j] * hp; vp += 1; j += 1
```

**Checklist:**

- [ ] Sắp **giảm dần** (đắt trước)?
- [ ] Cắt ngang nhân với số dải **cột** (`vp`), cắt dọc nhân với số dải **hàng** (`hp`)?
- [ ] Xử lý khi một mảng rỗng (`m = 1` hoặc `n = 1`)?
- [ ] Kết quả tới `~10^{13}` (64-bit ở ngôn ngữ khác)?